# Assignment 3

## Introduction
The goal of this visual analysis was to look at the cause of Tree Cover losses around the globe, but mainly in North America.
My interests stems from the desire to understand what are the factors and if wildfires play a key role.

### Instructions to run this dashboard:
**Pre-requisites**: This dashboard uses Panel and hvplots which requires you to run it locally from your computer where python and Jupyter are installed, along with the necessary python and visualization libraries.  

(1) Install the required python libraries by opening a terminal window on your computer and using the following command:

**pip install panel hvplot pandas numpy jupyter_bokeh plotly**

(3) Next, bring in the data files in the same directory as this notebook:

**'tree-cover-loss-by-dominant-driver.csv', 'wildfire_image.png'**

These files can be found on my github site at https://github.com/dqjohnson/umich/tree/Data-Visualization:

(4) Once in the Jupyter environment, to run the dashboard, do the following:

Run ALL the **code cells** in the notebook.  The dashboard will appear in a new tab in your web browser.  Use the widgets that contain the various variables (tree loss factor, year, location) on the left hand side bar to look at the data that appears in the charts on the right.

## Importing the necessary Libraries

In [36]:
import panel as pn                # The main dashboard framework
import pandas as pd              # For data manipulation
import numpy as np               # For numerical operations
import seaborn as sns            # For additional plotting capabilities
import hvplot.pandas             # Adds plotting methods directly to pandas DataFrames
import plotly.express as px
import holoviews as hv

# Initialize Panel extension - this is crucial!
# It enables Jupyter to display Panel objects and interactive widgets
pn.extension()
pn.extension('tabulator')
pn.extension('plotly')
hv.extension('bokeh')


## Demonstration

### Visualization technique: 
Line Plot: I wanted to track changes over time (e.g., to helps to see trends over the years).
Box Plot: I used these to compare the distributions across categories (e.g., tree loss by cause: wildfire vs. urbanization, etc.). These make great side-by-side comparisons.
Pie Chart: I wanted to show proportions against the whole (e.g., % of total tree coverage loss by cause).
Stacked Bar Chart: I wanted to show both total values of tree coverage loss and the composition of those totals. 
Bar Chart: I was able to compare individual categories side by side (e.g., tree coverage loss by country, for example)

Note: All the tree loss coverage is >= 30% loss

### Visualization Library:
I used Panel, hvplots functions, and plotly (for the pie chart because it was not provided in hvplots).  While harder than expected, panel provided a comprehensive set of tools to make nice looking visuals as well as simple widges to display my data.  Panel also came with a template to display my dashboard which made it much nicer. Panel and plotly also integrated with Jupyter, as well as provided the option of showing the dashboard in a web browser window. The downside is performance. When updating variables, there was a delay in rendering.

### Dataset
Data sources: Global Forest Watch (2024) https://www.globalforestwatch.org/ and Our World in Data: https://ourworldindata.org/

In [37]:
# cache data to improve dashboard performance
if 'data' not in pn.state.cache.keys():
    df = pd.read_csv('tree-cover-loss-by-dominant-driver.csv')
    pn.state.cache['data'] = df.copy()
else: 
    df = pn.state.cache['data']

#df.head()

In [38]:
df = df.fillna(0)
#Shorten column names
new_cols = ['location','code','year','commodity-driven deforestation','forestry','shifting agriculture','wildfire','urbanization']
df.columns = new_cols

In [39]:
#Add additional colums for totals - totals column(s)
loss_factors = ['commodity-driven deforestation','forestry', 'shifting agriculture','wildfire','urbanization']  

df['other factors except wildfires'] = df[[loss_factors[0], loss_factors[1],loss_factors[2],  loss_factors[4]]].sum(axis=1)
df['total loss'] = df[[loss_factors[0], loss_factors[1],loss_factors[2], loss_factors[3], loss_factors[4]]].sum(axis=1)
df['% loss by wildfires'] = df['wildfire']/df['total loss']
#df.head(100)

In [40]:
#defining key variables

continents =  ['North America','South America','Africa','Europe', 'Asia','Oceania']
continents_world = ['World','North America','South America','Africa','Europe', 'Asia','Oceania']
north_america = ['United States', 'North America', 'Canada', 'Mexico']
north_america_world = ['United States', 'North America', 'Canada', 'Mexico', 'World']
all_years_df = df['year'].to_list()
all_years = list(np.unique(all_years_df))

min_year = int(df['year'].min())
max_year = int(df['year'].max())
avg_year = int(df['year'].median())

numeric_cols = list(df.select_dtypes(include=[np.number]).columns)
categorical_cols = list(df.select_dtypes(exclude=[np.number]).columns)

data_summary = {
    col: {
        'type': str(df[col].dtype),
        'missing': df[col].isna().sum(),
        'unique_values': len(df[col].unique())
    } for col in df.columns
}


In [41]:
year_selector =  pn.widgets.Select(
    name='Select Year', 
    options=all_years,
    value=max_year,
)

year_slider = pn.widgets.IntSlider(
    name='Choose the year',
    start=min_year,
    end=max_year,step=1,
    value=max_year, 
    bar_color='#2ea648',
)

loss_factor_radio = pn.widgets.RadioButtonGroup(
    name='loss factor',
    options=loss_factors,
    button_type='primary', button_style='outline',
    description='Select the loss factor'
)

loss_factor_selector = pn.widgets.Select(
    name='Select Loss Factor', 
    options=loss_factors,
    value='wildfire',
)

location_selector = pn.widgets.Select(
    name='Select Location',
    options=north_america,
    value=north_america[0] if north_america else None,
    description='Select the location'
)



## Defining plot functions

Using 5 different plotting functions: line plot, box plot, bar plot, stacked bar plot and pie chart:

In [42]:
@pn.depends(loss_factor_selector)
def line_plot(loss_factor_selector):
    filtered_df = (
        df[
            (df.location.isin(north_america))
        ]
        .groupby(['location', 'year'])[loss_factor_selector]
        .sum()
        .reset_index()
    )

    plot = filtered_df.hvplot(
        x='year',
        y=loss_factor_selector,
        by='location',
        height=400,
        title=f'Tree Cover Loss due to "{loss_factor_selector}" (in hectares)',
        alpha=0.7,                       
        width=800,                  
        line_width=2,              
    )
    return plot



In [43]:
@pn.depends(loss_factor_selector)
def box_plot(loss_factor_selector):
    filtered_df = (
        df[
            (df.location.isin(north_america))
        ]
        .groupby(['location', 'year'])[loss_factor_selector]
        .sum()
        .reset_index()
    )

    plot = filtered_df.hvplot.box(
        y=loss_factor_selector,
        by='location',
        height=400,
        box_fill_color='Category',  # Color boxes by category
        whisker_color='black',      # Make whiskers black for contrast
        title=f'Tree Cover Loss due to: {loss_factor_selector}',
        box_alpha=0.7,                       
        width=700,                  
        legend='top',               
    )
    return plot
    

In [44]:
@pn.depends(year_slider,loss_factor_selector)
def bar_plot(year_slider,loss_factor_selector):
    filtered_df = (
        df[
            (df.location.isin(north_america_world)) &
            (df['year'] == year_slider)
        ]
        .groupby(['location'])[loss_factor_selector]
        .mean()
        .reset_index()
    )

    plot = filtered_df.hvplot.bar(
        x='location',
        y=loss_factor_selector,
        height=400,
        stacked=True,
        ylabel='Loss in Hectares',
        xlabel='Location',
        title=f'Total tree Cover Loss due to {loss_factor_selector} (in hectares)',
        alpha=0.7,                       
        width=500,                  
        rot=45, 
    )
    return plot
    

In [45]:
@pn.depends(year_selector,loss_factor_selector)
def stacked_bar_plot(year_selector,loss_factor_selector):
    filtered_df = (
        df[
            (df.location.isin(north_america_world)) &
            (df['year'] ==year_selector)
        ]
        .groupby(['location'])
        .sum()
        .reset_index()
    )

    plot = filtered_df.hvplot.bar(
        x='location',
        y=loss_factors,
        height=400,
        stacked=True,
        ylabel='Loss in Hectares',
        xlabel='Location',
        title=f'Tree Cover Loss for {year_selector}',
        alpha=0.7,                       
        width=800,                  
        rot=45,   
        legend='right'
    )
    return plot
    


In [46]:
@pn.depends(year_selector, location_selector)
def pie_chart_plot(year_selector, location_selector):
    filtered_df = (
        df[(df['year'] == year_selector) & 
        (df['location'] == location_selector)]
    )
    values = filtered_df[loss_factors].sum()
    pie_data = values.reset_index()
    pie_data.columns = ['Loss Factor', 'Hectares']

    plot = px.pie(
        pie_data,
        names='Loss Factor',
        values='Hectares',
        hole=0.2,
        title=f'Tree Cover Loss Factor Distribution in {location_selector}: {year_selector}',
        color_discrete_sequence=px.colors.qualitative.Antique
    )
    return plot


## Creating the Dashboard

Using a class to define the dashboard, and calling it using the class-defined function called view, which calls all my plot functions. Then using .show to make it appear in a browser window. 

In [47]:
import panel as pn

pn.extension()

class ResponsiveTabbedDashboard:

    def __init__(self, widgets, plots):
        self.widgets = widgets
        self.plots = plots
        self._layout = self.create_layout()

    def create_layout(self):
        sidebar= pn.Column(
            *self.widgets,
                pn.pane.Image('wildfire_image.png', width=300),
                pn.pane.Markdown('Wildfires have had a dramatic impact on our global society, particularly on Tree cover density, and are increasing \
                 in frequency and intensity. Tree cover is critical for reducing air pollution, and stabilizing biodiversity.'))

        
        tabs = pn.Tabs(
            ('Overview', pn.Column(
                pn.Row(pn.Column(self.plots['line_plot'], self.plots['box_plot'], sizing_mode='stretch_both')),
                pn.Row(pn.Column(self.plots['stacked_bar_plot'], self.plots['bar_plot'], self.plots['pie_chart_plot'], sizing_mode='stretch_both')),
            )),
            ('Statistics', pn.Column(pn.pane.DataFrame(self.plots['stats']), sizing_mode='stretch_both')),
            sizing_mode='stretch_both'
        )

        main_layout = pn.Column(tabs, sizing_mode='stretch_both')

        template = pn.template.FastListTemplate(
            title="'Dashboard: Tree Cover Loss due to a variety of factors",
            sidebar=[sidebar],
            main=[main_layout],
        )
        return template.servable();

    def view(self):
        return self._layout

dashboard = ResponsiveTabbedDashboard(
    widgets=[loss_factor_selector, location_selector, year_selector],
    plots={
        'pie_chart_plot': pie_chart_plot,
        'line_plot': line_plot,
        'box_plot': box_plot,
        'stacked_bar_plot': stacked_bar_plot,
        'bar_plot': bar_plot,
        'stats': pd.DataFrame(data_summary)
    }
)

dashboard.view().show()

Launching server at http://localhost:52650


ERROR:bokeh.core.validation.check:E-1001 (BAD_COLUMN_NAME): Glyph refers to nonexistent column name. This could either be due to a misspelling or typo, or due to an expected column being missing. : fill_color='Category' [no close matches] {renderer: GlyphRenderer(id='cfdd9f65-3417-4757-aa78-88ce624edd04', ...)}
ERROR:bokeh.core.validation.check:E-1001 (BAD_COLUMN_NAME): Glyph refers to nonexistent column name. This could either be due to a misspelling or typo, or due to an expected column being missing. : fill_color='Category' [no close matches] {renderer: GlyphRenderer(id='1ecad14e-dad3-48bd-b7e5-2a1ac8b50943', ...)}
